**Config**


In [0]:
from pyspark.sql.functions import col, concat_ws, explode_outer, from_json, lower, upper, replace, regexp_replace, concat,lit, current_date, expr, translate, to_date, current_timestamp
from pyspark.sql.types import StructType, StructField, StringType, ArrayType, LongType

catalogo = f"`oc_projeto`"
schema = f"`oc-tabelas`"
table_bronze = f'{catalogo}.{schema}.flights_bronze'
table_silver = f'{catalogo}.{schema}.silver_voos_dep'

In [0]:
print("Lendo dados da Bronze")
# Ler apenas últimos N dias:
df_bronze = spark.table(table_bronze).filter(
    col("data_carregamento") >= current_date() - expr("INTERVAL 7 DAYS")
)

In [0]:
df_despachados = df_bronze.select(
    
    col("turno"),
    col("data"),
    col("agente"),
    explode_outer(col("voos_despachados")).alias("voos_despachados")    
)

In [0]:
df_despachados_flat = df_despachados.select(
    col("turno"),
    col("data"),
    col("agente"),
    col("voos_despachados.*") #desaninhando a coluna
)

In [0]:
# Schema da coluna de AF chegando que está em string para JSON
# Exploded JSON

# Definir o schema do AF_chegando
af_saindo_schema = StructType([
    StructField("destino", StringType()),
    StructField("origens", ArrayType(StructType([
        StructField("voo", StringType()),
        StructField("origem", StringType()),
        StructField("qtd", LongType())
    ])))
])


In [0]:
df_despachados_flat2 = df_despachados_flat.select(
    col("turno"),
    col("data"),
    col("agente"),
    col("voo"),
    col("iata"),
   # Converter AF_chegando
    from_json(col("AF_saindo"), af_saindo_schema).alias("AF_saindo"),
    col("backtracking_feito"),
    col("bagagens_qtd"),
    col("cmt_tablet"),
    col("peso_bagagem_final"),
    col("peso_bagagem_inicial"),
    col("peso_carga_final"),
    col("peso_carga_inicial"),
    col("motivo_corte"),
    col("staff_qtd")
)

In [0]:
df_despachados3 = df_despachados_flat2.select(
    col("turno"),
    col("data"),
    col("agente"),
    col("voo"),
    col("iata"),
    
    col("AF_saindo.origens").alias("AF_origens"),
    col("AF_saindo.destino").alias("af_dep"),
    col("bagagens_qtd"),
    col("cmt_tablet"),
    col("peso_bagagem_final"),
    col("peso_bagagem_inicial"),
    col("peso_carga_final"),
    col("peso_carga_inicial"),
    col("motivo_corte"),
    col("staff_qtd")
    )
    

In [0]:
df_explode = df_despachados3.select(
    
    col("turno"),
    col("data"),
    col("agente"),
    col("voo").alias('voo_dep'),
    col("iata").alias('iata_dep'),
    explode_outer(col("AF_origens")).alias('af_origens'),
    col("af_dep").alias('af_dep'),
    col("bagagens_qtd").alias("bag_qtd"),
    col("cmt_tablet"),
    col("peso_bagagem_final"),
    col("peso_bagagem_inicial"),
    col("peso_carga_final"),
    col("peso_carga_inicial"),
    col("motivo_corte"),
    col("staff_qtd")
    )

In [0]:
df_destinos_flat = df_explode.select(
    lower(col("turno")).alias('turno'),
    to_date(col("data"), "dd/MM/yyyy").alias("data"),
    col("agente"),
    col("voo_dep"),
    col("iata_dep"),
    col("af_origens.voo").alias("af_origens_voo"),
    col("af_origens.origem").alias("af_origens_iata"),
    col("af_dep"),
    col("af_origens.qtd").alias("qtd"),
    col("bag_qtd"),
    lower(translate(col("cmt_tablet"), "áàãâäéèêëíìîïóòõôöúùûüç", "aaaaaeeeeiiiiooooouuuuc")).alias("cmt_tablet"),
    col("peso_bagagem_final"),
    col("peso_bagagem_inicial"),
    col("peso_carga_final"),
    col("peso_carga_inicial"),
    lower(col("motivo_corte")).alias('motivo_corte'),
    col("staff_qtd")
)

In [0]:
from pyspark.sql.functions import when

df_destinos_flat = df_destinos_flat.withColumn(
    "af_origens_voo",
    when(
            (col("af_origens_voo") == "111") | (col("af_origens_voo") == "1111"),
            col("af_dep")
        ).otherwise(col("af_origens_voo"))
    )

In [0]:
# Criando coluna ID unico.

df_com_id = df_destinos_flat.select(
    concat_ws("_",
        regexp_replace(col("data"), "/", ""),
        col("agente"),
        concat(lit("o"), col("af_origens_voo")),
        col("af_origens_iata"),
        concat(lit("d"),col("voo_dep")),
        col("iata_dep"),
    ).alias("id_voo"),
    col("*")
)

In [0]:
df_com_id = df_com_id.withColumn("data_processamento", current_timestamp())

df_com_id.createOrReplaceTempView("temp_silver_dep")

spark.sql(f"""
    MERGE INTO {table_silver} AS target
    USING temp_silver_dep AS source
    ON target.id_voo = source.id_voo
    
    WHEN NOT MATCHED THEN INSERT *
""")